# Qwen3.8-27B on Modal — no Kaggle queue, pay per container-second 🚀

Kaggle's free TPU is great but queues (3–9 min mornings, 80–95 min evenings) and caps sessions at
9 hours. [Modal](https://modal.com) is the complement: cold starts in seconds-to-minutes, billed per
container-second ($30/month free credit at signup), and scales to zero when idle.

This notebook drives the appliance in the repo's `modal/` folder: **Qwen3.8-27B GPTQ-Int4**
(~19 GiB of weights in a Modal Volume) behind vLLM with an API key. Measured reference points
(from the Burst-Compute-Testing campaigns): **A100-40GB full profile** (graphs on, c1 ~55 tok/s,
p50 TTFT ~215 ms) and **A10G 24 GiB** with a shrunken profile (ctx 6144, eager). **T4 measured OOM** —
the artifact dies 86 MiB short; do not use it. L40S is configured (full 262K context) but requires
a payment method on file.

## Before you run
1. `pip install modal` (the first cell does it)
2. `modal setup` — or run `!modal token new` in the second cell and follow the browser prompt
3. The one-time weights download (~19 GiB) runs in cell 3; it goes into a Modal Volume so later
   boots never re-download it.

Then run the cells top to bottom. The deploy cell prints your public URL; the secret cell makes
sure it is not open to the internet.

In [ ]:
%pip install -q modal
import modal
print("modal CLI ready")

In [ ]:
# One-time: authenticate. Opens a browser link; paste the token back.
!modal token new

In [ ]:
# One-time (~19 GiB): download the GPTQ-Int4 checkpoint into a Modal Volume so the
# serving app boots offline. Run the script that ships next to this notebook.
# (From a checkout of this repo:  python modal/populate_qwen38_volume.py )
!python modal/populate_qwen38_volume.py

## Set an API key
The endpoint Modal prints is public; vLLM's `--api-key` middleware is the auth layer.
Create a Modal secret that carries your key (and any deploy knobs) once:

```bash
modal secret create qwen38-serve-key BURST_SERVE_KEY=sk-your-secret
```

Pick a long random key (`python -c "import secrets; print('sk-'+secrets.token_hex(16))"`).
Treat the URL+key pair like a credential; a redeploy keeps the secret, a new one needs a new key.

In [ ]:
# Deploy. Default profile: L40S (48 GiB), full 262,144-token context — needs a payment
# method on file. On free credits, use the A10G profile instead (second command):

# !modal deploy modal/serve_qwen38_modal.py

# A10G free-credit profile (shrunken: 6144 ctx, eager, 4 seqs):
!BURST_GPU=A10G BURST_MAX_MODEL_LEN=6144 BURST_GPU_MEM_UTIL=0.98 \
  BURST_MAX_NUM_SEQS=4 BURST_MAX_NUM_BATCHED_TOKENS=1024 BURST_ENFORCE_EAGER=1 \
  modal deploy modal/serve_qwen38_modal.py

## Use the endpoint
The deploy output prints a URL like
`https://<workspace>--burst-qwen38-serve.us-east.modal.direct/v1`.

```bash
curl https://<workspace>--burst-qwen38-serve.us-east.modal.direct/v1/chat/completions \
  -H "Authorization: Bearer sk-your-secret" -H "Content-Type: application/json" \
  -d '{"model": "qwen38-27b", "messages": [{"role": "user", "content": "Hello!"}]}'
```

Point any OpenAI-compatible client at it:
```bash
export OPENAI_BASE_URL=https://<workspace>--burst-qwen38-serve.us-east.modal.direct/v1
export OPENAI_API_KEY=sk-your-secret
```

## Cost discipline
- The container scales to zero after `BURST_SCALEDOWN_SECONDS` (default 600 s) idle — GPU
  billing stops with it.
- End a deployment for good with `!modal app stop burst-qwen38`.
- Check spend with `!modal billing summary`.
- Idle GPU seconds burn credits just like idle TPU minutes burn Kaggle quota.